# Session 8 · LangChain essentials
Run from top to bottom. Each cell teaches one small idea. Only cells that call
the model use API quota. Outputs vary; no answer is sent to a customer.


## 0. Setup


**Concept** · Imports make library tools available in Python. Nothing is sent to Gemini here.

**What this cell does** · Loads the tools used in the examples.

**What to look for** · No output means the imports worked.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 0 · Import the building blocks      │
# └──────────────────────────────────────────┘
import json
import os
import sys
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.exceptions import OutputParserException
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, CommaSeparatedListOutputParser
from langchain_core.runnables import Runnable, RunnableLambda, RunnableSequence, RunnableParallel


**Concept** · A configuration file stores a setting in one place. Both exercises read the same model ID.

**What this cell does** · Finds the repository root from either notebook launch directory.

**What to look for** · The selected model ID, never your key.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 0 · Find the shared model setting   │
# └──────────────────────────────────────────┘
root = Path.cwd()
if not (root / "workshop_config.py").exists():
    root = root.parent
if not (root / "workshop_config.py").exists():
    raise RuntimeError("Launch Jupyter from the repository root or notebook folder.")
sys.path.insert(0, str(root))
from workshop_config import MODEL_ID
print("Model:", MODEL_ID)


**Concept** · A key authorizes requests. A .env file keeps it out of the notebook.

**What this cell does** · Loads your local key and creates a Gemini chat model without calling it.

**What to look for** · A friendly error if the key is missing. The key itself is never displayed.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 0 · Load the key and create a model │
# └──────────────────────────────────────────┘
load_dotenv(find_dotenv())
key = os.getenv("GOOGLE_API_KEY", "").strip()
if not key or key == "your-gemini-key-here":
    raise RuntimeError("Add GOOGLE_API_KEY to the root .env file, then rerun this cell.")
model = init_chat_model(f"google_genai:{MODEL_ID}", vertexai=False)
print("Model ready. No API call yet.")


## 1. What is LangChain?
LangChain gives different models the same interface. It also connects prompts,
models, and parsers. The expression `a | b` means “give a's result to b.” This
composition style is called LCEL (LangChain Expression Language).

These are **reading examples**, not extra cells to execute. Each snippet assumes
the matching provider key is configured locally. We only use Gemini in this class.

| Raw Gemini SDK | LangChain |
| --- | --- |
| `from google import genai` | `from langchain.chat_models import init_chat_model` |
| `client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])` | `model = init_chat_model(f"google_genai:{MODEL_ID}")` |
| `reply = client.models.generate_content(model=MODEL_ID, contents="Hello")` | `reply = model.invoke("Hello")` |
| `print(reply.text)` | `print(reply.text)` |

Other providers have their own SDKs. These examples are markdown only and require
separate keys and packages; do not run them for this workshop.

```python
# OpenAI SDK — reading only
from openai import OpenAI
client = OpenAI()
result = client.responses.create(model="YOUR_OPENAI_MODEL", input="Hello")
print(result.output_text)
```

```python
# Claude SDK — reading only
from anthropic import Anthropic
client = Anthropic()
result = client.messages.create(
    model="YOUR_CLAUDE_MODEL", max_tokens=100,
    messages=[{"role": "user", "content": "Hello"}],
)
```


## 2. What is a Runnable?


**Concept** · Runnables do a job. Messages are the data passed between those jobs.

**What this cell does** · Checks prompts, a model, a parser, a chain, a Python function wrapper, and a message.

**What to look for** · Five True values followed by False for the HumanMessage.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 2 · Jobs versus their input data    │
# └──────────────────────────────────────────┘
prompt = ChatPromptTemplate.from_template("Explain {topic} in one sentence.")
parser = StrOutputParser()
chain = prompt | model | parser
shout = RunnableLambda(lambda text: text.upper())
examples = [("prompt", prompt), ("model", model), ("parser", parser),
            ("chain", chain), ("lambda", shout), ("message", HumanMessage("Hello"))]
for name, item in examples:
    print(name, isinstance(item, Runnable))


## 3. What the model returns


**Concept** · The API sends JSON over the network. LangChain turns that response into an AIMessage object.

**What this cell does** · Makes one request and prints text plus metadata, which is information about the reply.

**What to look for** · Text, token counts when supplied, provider details, and an ID.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 3 · Inspect an AIMessage            │
# └──────────────────────────────────────────┘
reply = model.invoke("Explain a Runnable in one short sentence.")
print(type(reply).__name__)
print(reply.text)
print(reply.usage_metadata)
print(reply.response_metadata)
print(reply.id)


## 4. Messages


**Concept** · A SystemMessage sets behavior. A HumanMessage supplies the question or task.

**What this cell does** · Asks the same question with two different system instructions; this makes two requests.

**What to look for** · How the tone or detail changes while the question stays the same.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 4 · Change the system instruction   │
# └──────────────────────────────────────────┘
question = HumanMessage("How should I prepare for a day hike?")
for instruction in ["Use one playful sentence for a child.",
                    "Use three concise safety tips for an adult."]:
    answer = model.invoke([SystemMessage(instruction), question])
    print(instruction)
    print(answer.text, "\n")


## 5. Prompt templates


**Concept** · A template is text with named blanks. invoke fills those blanks locally.

**What this cell does** · Builds a PromptTemplate and prints its filled value.

**What to look for** · No Gemini request; only the completed prompt.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 5 · Fill a string template          │
# └──────────────────────────────────────────┘
plain_prompt = PromptTemplate.from_template("Describe {product} in {words} words.")
plain_value = plain_prompt.invoke({"product": "hiking socks", "words": 12})
print(plain_value)
print(plain_value.to_string())


**Concept** · ChatPromptTemplate preserves message roles. MessagesPlaceholder inserts earlier conversation messages.

**What this cell does** · Adds a short history before a new question, without calling the model.

**What to look for** · System, previous human/AI, and current human messages in order.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 5 · Fill a chat template            │
# └──────────────────────────────────────────┘
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You help hikers choose equipment."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
history = [HumanMessage("I hike in rain."), AIMessage("Bring a waterproof layer.")]
chat_value = chat_prompt.invoke({"history": history, "question": "What about shoes?"})
print(chat_value)


**Concept** · Blank names must match exactly. Double braces mean literal braces rather than a blank.

**What this cell does** · Catches a misspelled input key and then prints a template containing literal JSON.

**What to look for** · A caught KeyError and a normal single pair of braces in the final text.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 5 · Catch a typo; escape braces     │
# └──────────────────────────────────────────┘
try:
    plain_prompt.invoke({"prodcut": "boots", "words": 12})
except KeyError:
    print("Caught KeyError: use product, not prodcut.")
braces = PromptTemplate.from_template('Return {{"product": "{product}"}}')
print(braces.invoke({"product": "boots"}).to_string())


## 6. Output parsers


**Concept** · A parser changes the shape of an answer. StrOutputParser extracts message text.

**What this cell does** · Parses the AIMessage saved in section 3 without another API call.

**What to look for** · A Python str containing the same reply text.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 6 · Convert a message to a string   │
# └──────────────────────────────────────────┘
text_answer = StrOutputParser().invoke(reply)
print(type(text_answer).__name__)
print(text_answer)
assert text_answer == reply.text


## 7. Chains and the pipe


**Concept** · A chain passes each result into the next step. The pipe | connects compatible steps.

**What this cell does** · Builds and invokes a three-step chain.

**What to look for** · The final result is a string.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 7 · Connect prompt, model, parser   │
# └──────────────────────────────────────────┘
prompt = ChatPromptTemplate.from_template("Explain {topic} in one sentence.")
chain = prompt | model | StrOutputParser()
print(chain.invoke({"topic": "waterproof jackets"}))


**Concept** · Intermediate results help you find the stage that caused a problem.

**What this cell does** · Prints prompt alone, prompt plus model, then the full chain; the last two make separate requests.

**What to look for** · A prompt value, an AIMessage, and a string. The two replies may differ.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 7 · Debug one stage at a time       │
# └──────────────────────────────────────────┘
inputs = {"topic": "trail snacks"}
print(prompt.invoke(inputs))
message_stage = (prompt | model).invoke(inputs)
print(type(message_stage).__name__, message_stage.text)
print(chain.invoke(inputs))


## 8. invoke versus batch versus stream
`invoke` handles one input. `batch` handles a list of inputs and returns results
in the same order. `stream` yields pieces as they become available. Concurrency
means overlapping work; a low limit helps avoid free-tier 429 rate-limit errors.
If you hit a limit, wait and retry the cell later, or lower concurrency to 1.


**Concept** · The same Runnable supports one request or several. Batch keeps input order.

**What this cell does** · Invokes one input, then batches two inputs with at most two overlapping items.

**What to look for** · One string and a list of two strings. This makes three model requests.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 8 · One input and a small batch     │
# └──────────────────────────────────────────┘
print(chain.invoke({"topic": "hiking boots"}))
topics = [{"topic": "headlamps"}, {"topic": "backpacks"}]
answers = chain.batch(topics, config={"max_concurrency": 2})
for inputs, answer in zip(topics, answers):
    print(inputs["topic"], "→", answer)


**Concept** · Streaming lets you show text before the whole answer is ready.

**What this cell does** · Streams directly from the model and reads each message chunk with .text.

**What to look for** · Pieces appear as they arrive; buffering can make several appear together.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 8 · Print streamed message pieces   │
# └──────────────────────────────────────────┘
for chunk in model.stream("Give three short tips for packing a daypack."):
    print(chunk.text, end="", flush=True)
print()


## 9. Structured output


**Concept** · JSON is a strict text format. Chatter and markdown fences around JSON are not valid JSON.

**What this cell does** · Parses a hard-coded reply with chatter and code fences; no API call is made.

**What to look for** · A caught JSONDecodeError rather than a crashed notebook.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 9a · Why plain json.loads can fail  │
# └──────────────────────────────────────────┘
chatty_reply = 'Here you go!\n```json\n{"sentiment": "positive"}\n```'
try:
    print(json.loads(chatty_reply))
except json.JSONDecodeError:
    print("json.loads expects only JSON, without chatter or fences.")


**Concept** · JsonOutputParser asks for JSON and converts the reply to a dictionary. It does not enforce field rules by itself.

**What this cell does** · Inserts get_format_instructions into a prompt and runs a JSON chain.

**What to look for** · A dictionary. Asking for JSON is less reliable than native structured output.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 9b · Ask for JSON, then parse it    │
# └──────────────────────────────────────────┘
json_parser = JsonOutputParser()
json_prompt = ChatPromptTemplate.from_template(
    "Extract sentiment and issue from this review: {review}\n{format_instructions}"
).partial(format_instructions=json_parser.get_format_instructions())
json_chain = json_prompt | model | json_parser
print(json_chain.invoke({"review": "These socks are warm and comfortable."}))


**Concept** · A comma-separated list is useful for simple answers. A parser can turn it into a Python list.

**What this cell does** · Adds list format instructions and parses three suggested items.

**What to look for** · A list of strings, rather than one long string.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 9c · Parse a comma-separated list   │
# └──────────────────────────────────────────┘
list_parser = CommaSeparatedListOutputParser()
list_prompt = ChatPromptTemplate.from_template(
    "Name three day-hike essentials. {format_instructions}"
).partial(format_instructions=list_parser.get_format_instructions())
list_chain = list_prompt | model | list_parser
print(list_chain.invoke({}))


### A plain dictionary schema
A schema describes the shape of data. **enum = a fixed list of allowed answers**;
here sentiment can only be positive, neutral, or negative. `required` lists fields
that must be present. `issue` uses the string `none` when there is no complaint.
Gemini's native structured output constrains generation. A plain dict still needs
Python-side checks if you rely on its values; the lab shows small checks.


**Concept** · A JSON schema is an ordinary nested Python dictionary. No Pydantic class is needed.

**What this cell does** · Defines three required review fields and the allowed sentiment values.

**What to look for** · The required field names. This cell is local.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 9d · Describe the expected fields   │
# └──────────────────────────────────────────┘
review_schema = {
    "title": "ReviewFacts", "description": "Facts from one review.",
    "type": "object",
    "properties": {
        "sentiment": {"type": "string", "enum": ["positive", "neutral", "negative"]},
        "product": {"type": "string"},
        "issue": {"type": "string", "description": "Problem, or 'none'."},
    },
    "required": ["sentiment", "product", "issue"],
}
print(review_schema["required"])


**Concept** · with_structured_output returns a Runnable that asks Gemini to follow the schema.

**What this cell does** · Uses the plain dictionary schema, then checks required values locally.

**What to look for** · A dictionary with sentiment, product, and issue.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 9d · Request native structured data │
# └──────────────────────────────────────────┘
structured_model = model.with_structured_output(review_schema, method="json_schema")
facts = structured_model.invoke("Review of hiking socks: warm and comfortable, no complaint.")
assert all(isinstance(facts.get(field), str) and facts[field].strip()
           for field in review_schema["required"])
assert facts["sentiment"] in review_schema["properties"]["sentiment"]["enum"]
print(facts)


**Concept** · A parser can fail even when a request succeeds. Catch its specific exception so you can recover.

**What this cell does** · Feeds a deliberately invalid hard-coded reply to JsonOutputParser.

**What to look for** · A caught OutputParserException. No model request is made.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 9e · Catch a parser failure         │
# └──────────────────────────────────────────┘
try:
    JsonOutputParser().invoke(AIMessage("This is not JSON."))
except OutputParserException:
    print("Could not parse the reply. Ask again or route it to a human.")


**Concept** · include_raw keeps the message alongside the parsed result and any parsing error. Network and quota failures still raise.

**What this cell does** · Invokes a structured model and displays parsed and parsing_error.

**What to look for** · Usually parsing_error is None. This option reports errors; it does not retry or validate every business rule.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 9e · Keep the raw reply for diagnosis │
# └──────────────────────────────────────────┘
diagnostic_model = model.with_structured_output(
    review_schema, method="json_schema", include_raw=True
)
diagnostic = diagnostic_model.invoke("Review of a headlamp: bright and reliable.")
print("parsed:", diagnostic["parsed"])
print("parsing_error:", diagnostic["parsing_error"])
print("raw type:", type(diagnostic["raw"]).__name__)


## 10. RunnableSequence and RunnableParallel


**Concept** · A sequence runs steps in order because the next step needs the previous result.

**What this cell does** · Builds a summary step a and a headline step b. A lambda packages the summary into b’s named blank.

**What to look for** · Two reusable chains; creating them does not call the API.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 10 · Build dependent steps          │
# └──────────────────────────────────────────┘
summary_prompt = ChatPromptTemplate.from_template("Summarize in one sentence: {text}")
headline_prompt = ChatPromptTemplate.from_template("Write a short headline for: {summary}")
a = summary_prompt | model | StrOutputParser()
b = (RunnableLambda(lambda summary: {"summary": summary})
     | headline_prompt | model | StrOutputParser())
pipe_sequence = a | b
explicit_sequence = RunnableSequence(a, b)


**Concept** · The pipe and RunnableSequence express the same data flow. Generated words can still differ across runs.

**What this cell does** · Runs both versions on the same input, using two requests per version.

**What to look for** · A headline from each version. Both summarize first, then write a headline.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 10 · Two spellings of a sequence    │
# └──────────────────────────────────────────┘
story = {"text": "A local hiking group collected litter and repaired trail signs on Saturday."}
print("pipe:", pipe_sequence.invoke(story))
print("explicit:", explicit_sequence.invoke(story))


**Concept** · Parallel branches receive the same input. Their outputs are collected under names you choose.

**What this cell does** · Drafts a reply and labels urgency independently for the same review.

**What to look for** · A dictionary with reply and urgency keys. These branches each call Gemini.


In [ ]:
# ┌──────────────────────────────────────────┐
# │ STEP 10 · Run independent jobs in parallel │
# └──────────────────────────────────────────┘
reply_prompt = ChatPromptTemplate.from_template("Draft a polite short reply to: {review}")
urgency_prompt = ChatPromptTemplate.from_template(
    "Classify urgency as low, medium, or high. Safety risks are high. Review: {review}"
)
reply_chain = reply_prompt | model | StrOutputParser()
urgency_chain = urgency_prompt | model | StrOutputParser()
parallel = RunnableParallel(reply=reply_chain, urgency=urgency_chain)
print(parallel.invoke({"review": "My stove valve sticks open and leaks fuel."},
                      config={"max_concurrency": 2}))


### Which one do I use?

| Need | Choose | Why |
| --- | --- | --- |
| One step needs another's output | `a | b` or `RunnableSequence(a, b)` | They run in order |
| Independent jobs use the same input | `RunnableParallel(reply=a, urgency=b)` | Branch results are named |
| The same job handles many inputs | `chain.batch(inputs, config={"max_concurrency": 2})` | Returns one result per input |

The urgency above is a plain text demonstration. The lab uses an enum schema so
the sorting code receives only low, medium, or high. Parallel branches can overlap
inside each batch item, so batch concurrency is not a global request-rate limit.

## 11. Recap

- Runnables do jobs; messages carry data between jobs.
- Templates fill named blanks locally before any model request.
- Models return AIMessage objects; read reply text with `.text`.
- Pipes connect steps; sequences depend on earlier results and parallel branches share inputs.
- Parsers change output shapes; schemas and local checks make structured values safer to use.
